# Bug hunt: the answer key

Each stage shows the planted bug, what it does, and the corrected code with its numbers.

In [1]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.neighbors import KNeighborsClassifier
import torch
from torch import nn

## Stage 1: choosing the useful features

The table has 200 patients and 500 measurements; most are noise. We keep the 10 measurements that relate to the outcome, then evaluate a model.

In [2]:
rng = np.random.default_rng(0)
X = rng.normal(size=(200, 500)); y = rng.integers(0, 2, 200)                    # in truth the outcome is unrelated to every measurement
X_best = SelectKBest(f_classif, k=10).fit_transform(X, y)                       # keep the 10 measurements most related to the outcome
X_train, X_test, y_train, y_test = train_test_split(X_best, y, test_size=0.3, random_state=0)
model = LogisticRegression().fit(X_train, y_train)
print("test accuracy:", round(model.score(X_test, y_test), 3))

test accuracy: 0.733


**The bug.** The feature selection looks at the labels of **all** 200 patients before the split, so the test patients helped choose the features. The outcome is pure noise, yet the test accuracy is far above 0.5. A selection step is a fitted step (Session 2): put it in a pipeline fitted on the training rows only.

In [3]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0)
honest = make_pipeline(SelectKBest(f_classif, k=10), LogisticRegression()).fit(X_train, y_train)      # selection learned from the training rows only
print("test accuracy:", round(honest.score(X_test, y_test), 3))

test accuracy: 0.417


## Stage 2: when to stop training

We train a small network for 60 epochs and report the test accuracy of the best epoch.

In [4]:
torch.manual_seed(0); rng = np.random.default_rng(1)
Xm = rng.normal(size=(300, 20)); ym = (Xm[:, 0] + 0.5 * rng.normal(size=300) > 0).astype(np.float32)
Xtr, Xte, ytr, yte = train_test_split(Xm, ym, test_size=0.3, random_state=0)
Xtr, Xte, ytr, yte = [torch.tensor(a, dtype=torch.float32) for a in (Xtr, Xte, ytr, yte)]
net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1)); opt = torch.optim.Adam(net.parameters(), lr=0.01)
best = 0
for epoch in range(60):
    loss = nn.functional.binary_cross_entropy_with_logits(net(Xtr).squeeze(1), ytr); opt.zero_grad(); loss.backward(); opt.step()
    acc = ((net(Xte).squeeze(1) > 0).float() == yte).float().mean().item()
    best = max(best, acc)                                                      # keep the best epoch
print("reported test accuracy:", round(best, 3))

reported test accuracy: 0.833


**The bug.** The best epoch is chosen by the **test** accuracy, so the reported number is the maximum of 60 noisy measurements on the same test set: it is biased upwards. Choose the epoch (early stopping) on a validation set taken from the training rows, and open the test set once at the end.

In [5]:
torch.manual_seed(0)
Xa, Xv, ya, yv = train_test_split(Xtr.numpy(), ytr.numpy(), test_size=0.25, random_state=0)           # a validation split taken from the TRAINING rows
Xa, Xv, ya, yv = [torch.tensor(a, dtype=torch.float32) for a in (Xa, Xv, ya, yv)]
net = nn.Sequential(nn.Linear(20, 64), nn.ReLU(), nn.Linear(64, 1)); opt = torch.optim.Adam(net.parameters(), lr=0.01)
best_val, kept = 0, None
for epoch in range(60):
    loss = nn.functional.binary_cross_entropy_with_logits(net(Xa).squeeze(1), ya); opt.zero_grad(); loss.backward(); opt.step()
    val = ((net(Xv).squeeze(1) > 0).float() == yv).float().mean().item()
    if val > best_val: best_val, kept = val, [p.detach().clone() for p in net.parameters()]
with torch.no_grad():
    for p, k in zip(net.parameters(), kept): p.copy_(k)
print("reported test accuracy:", round(((net(Xte).squeeze(1) > 0).float() == yte).float().mean().item(), 3))

reported test accuracy: 0.789


## Stage 3: a categorical column

Customers have a plan (basic, plus, pro). The plan matters a lot for the outcome. We encode it as numbers and fit a model.

In [6]:
rng = np.random.default_rng(2); n = 600
plan = rng.choice(["basic", "plus", "pro"], n, p=[0.5, 0.3, 0.2]); usage = rng.normal(size=n)
y = ((plan == "pro") * 2.0 + (plan == "basic") * -1.0 + 0.5 * usage + rng.normal(0, 0.5, n) > 0).astype(int)
frame = pd.DataFrame({"plan": plan, "usage": usage})
tr, te, ytr_c, yte_c = train_test_split(frame, y, test_size=0.3, random_state=1)
enc_tr = pd.factorize(tr["plan"])[0]; enc_te = pd.factorize(te["plan"])[0]                                          # encode the plan as 0, 1, 2
model = LogisticRegression().fit(np.c_[enc_tr, tr.usage], ytr_c)
print("test accuracy:", round(model.score(np.c_[enc_te, te.usage], yte_c), 3))

test accuracy: 0.278


**The bug.** `pd.factorize` numbers the categories in the order they first appear, separately in the training rows and in the test rows, so the same number means a different plan in the two sets. Fit the encoder once on the training rows and apply it to both. Worse, ordinal codes make 'pro' look like 'two plus basic'; a one-hot encoding is safer for a plan with no real order.

In [7]:
enc = OrdinalEncoder(categories=[["basic", "plus", "pro"]]).fit(tr[["plan"]])                                   # fitted once, on the training rows
model = LogisticRegression().fit(np.c_[enc.transform(tr[["plan"]]), tr.usage], ytr_c)
print("test accuracy:", round(model.score(np.c_[enc.transform(te[["plan"]]), te.usage], yte_c), 3))

test accuracy: 0.878


## Stage 4: nearest neighbours

We predict who will churn from two features: income in dollars and the number of support calls. We use k-nearest neighbours.

In [8]:
rng = np.random.default_rng(3); n = 500
income = rng.normal(50_000, 15_000, n); calls = rng.poisson(3, n).astype(float)
churn = (calls + rng.normal(0, 1.0, n) > 4).astype(int)                          # churn depends on the calls, not on the income
Xk = np.c_[income, calls]
Xk_tr, Xk_te, yk_tr, yk_te = train_test_split(Xk, churn, test_size=0.3, random_state=0)
knn = KNeighborsClassifier(n_neighbors=15).fit(Xk_tr, yk_tr)
print("test accuracy:", round(knn.score(Xk_te, yk_te), 3))

test accuracy: 0.733


**The bug.** k-NN works from distances, and income is measured in tens of thousands while calls are 0 to 10: the income decides every distance and the calls, the only useful feature, are ignored. Scale the features (inside a pipeline, so the scale comes from the training rows).

In [9]:
knn = make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=15)).fit(Xk_tr, yk_tr)
print("test accuracy:", round(knn.score(Xk_te, yk_te), 3))

test accuracy: 0.913


## Stage 5: predictions on new data

The network below uses dropout. We train it and then evaluate it on the held-out rows.

In [10]:
torch.manual_seed(0); rng = np.random.default_rng(4)
Xd = rng.normal(size=(400, 10)); yd = (Xd[:, :3].sum(1) > 0).astype(np.float32)
Xd_tr, Xd_te, yd_tr, yd_te = [torch.tensor(a, dtype=torch.float32) for a in train_test_split(Xd, yd, test_size=0.3, random_state=0)]
drop = nn.Sequential(nn.Linear(10, 128), nn.ReLU(), nn.Dropout(0.5), nn.Linear(128, 1)); opt = torch.optim.Adam(drop.parameters(), lr=0.01)
for epoch in range(150):
    drop.train(); loss = nn.functional.binary_cross_entropy_with_logits(drop(Xd_tr).squeeze(1), yd_tr); opt.zero_grad(); loss.backward(); opt.step()
accs = [((drop(Xd_te).squeeze(1) > 0).float() == yd_te).float().mean().item() for _ in range(5)]
print("test accuracy, five evaluations:", [round(a, 3) for a in accs])

test accuracy, five evaluations: [0.925, 0.942, 0.942, 0.933, 0.908]


**The bug.** The model is still in training mode, so dropout is switched on during evaluation: every evaluation of the **same** data gives a different answer, and a lower one. Call `model.eval()` (and use `torch.no_grad()`) before predicting; Session 7 notes that `evaluate` must switch dropout off.

In [11]:
drop.eval()                                                                      # dropout off
with torch.no_grad():
    accs = [((drop(Xd_te).squeeze(1) > 0).float() == yd_te).float().mean().item() for _ in range(5)]
print("test accuracy, five evaluations:", [round(a, 3) for a in accs])

test accuracy, five evaluations: [0.933, 0.933, 0.933, 0.933, 0.933]
